# Laboratorio AWS — Predicción de temperatura de un transformador
## Notebook 03 — Entrenamiento administrado con Amazon SageMaker

### Objetivo

Entender y preparar un **SageMaker Training Job** utilizando los datos almacenados en Amazon S3.

En la cuenta utilizada para este laboratorio, la cuota de `ml.m5.large` para Training Jobs es actualmente `0`. Por esta razón, el entrenamiento administrado se deja preparado, validado y documentado, pero **no se convierte en un requisito para continuar**.

### Flujo

```text
S3 / data/train + S3 / data/test
              ↓
       ModelTrainer
              ↓
     Training Job administrado
              ↓
       model.tar.gz
              ↓
            S3
```

### Estrategia del laboratorio

Si la cuota permite crear el Training Job, se puede ejecutar la celda marcada como **OPCIONAL — CÓMPUTO AWS**.

Si la cuota no lo permite, el notebook continúa usando el modelo que ya fue entrenado y validado en el Notebook 02.

> **Resultado importante:** una limitación de infraestructura no detiene el flujo pedagógico.

## Paso 1. Arquitectura y punto de partida

El Notebook 02 dejó disponibles:

- `data/train/train.csv`
- `data/test/test.csv`
- `model/mejor_modelo.joblib`
- `output/metricas_modelos.csv`
- `output/predicciones_test.csv`

El Notebook 03 introduce el concepto de entrenamiento administrado.

In [21]:
import os
import json
from pathlib import Path

import boto3
import pandas as pd
import sagemaker

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    SourceCode,
    Compute,
    InputData,
    StoppingCondition,
)
from sagemaker.core.shapes import OutputDataConfig
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role


print("SageMaker SDK cargado.")
print(f"Boto3: {boto3.__version__}")

## Paso 2. Configuración del proyecto

No se escriben credenciales AWS en el notebook.

SageMaker Studio utiliza el IAM Role asociado al entorno de ejecución.

In [22]:
BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

TRAIN_PREFIX = "data/train/"
TEST_PREFIX = "data/test/"
MODEL_PREFIX = "model/sagemaker-training/"
LOCAL_MODEL = Path("model_artifacts/mejor_modelo.joblib")

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

print(f"Región: {REGION}")
print(f"Bucket: {BUCKET}")

## Paso 3. Verificar datos y permisos

Esta sección solamente consulta S3.

No crea infraestructura de entrenamiento.

In [23]:
s3.head_bucket(Bucket=BUCKET)

for prefix in [TRAIN_PREFIX, TEST_PREFIX]:
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    keys = [item["Key"] for item in response.get("Contents", [])]
    print(f"{prefix}")
    for key in keys:
        print(" -", key)

## Paso 4. Crear el script de entrenamiento

El script representa lo que SageMaker ejecutaría dentro de su contenedor.

El modelo utilizado es `RandomForestRegressor`.

El script:

1. recibe `train.csv` y `test.csv`;
2. entrena el modelo;
3. calcula MAE, RMSE y R²;
4. guarda `model.joblib` en `/opt/ml/model/`.

In [24]:
training_code = r'''
import argparse
import json
import os
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "temperatura_transformador"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=10)
    args = parser.parse_args()

    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    test_dir = os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test")
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")

    train = pd.read_csv(os.path.join(train_dir, "train.csv"))
    test = pd.read_csv(os.path.join(test_dir, "test.csv"))

    X_train = train.drop(columns=[TARGET])
    y_train = train[TARGET]
    X_test = test.drop(columns=[TARGET])
    y_test = test[TARGET]

    model = RandomForestRegressor(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics = {
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": mean_squared_error(y_test, predictions) ** 0.5,
        "r2": r2_score(y_test, predictions)
    }

    print(json.dumps(metrics))

    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(model_dir, "model.joblib"))

if __name__ == "__main__":
    main()
'''

Path("train.py").write_text(training_code, encoding="utf-8")
compile(training_code, "train.py", "exec")
print("train.py creado y validado.")

## Paso 5. Preparar `ModelTrainer`

Esta sección **no crea un Training Job**.

Solamente construye la configuración que SageMaker utilizaría para ejecutar el entrenamiento administrado.

La instancia propuesta es `ml.m5.large`, una sola instancia y un máximo de 600 segundos.

In [25]:
TRAINING_INSTANCE = "ml.m5.large"

training_image = image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version="1.2-1",
    py_version="py3",
    instance_type=TRAINING_INSTANCE,
    image_scope="training"
)

role = get_execution_role()

source_code = SourceCode(
    source_dir=".",
    entry_script="train.py",
    ignore_patterns=[
        ".git", ".gitignore", ".ipynb_checkpoints",
        "__pycache__", ".env", "data", "model", "output"
    ]
)

compute = Compute(
    instance_type=TRAINING_INSTANCE,
    instance_count=1
)

stopping_condition = StoppingCondition(
    max_runtime_in_seconds=600
)

input_data_config = [
    InputData(
        channel_name="train",
        data_source=f"s3://{BUCKET}/{TRAIN_PREFIX}",
        content_type="text/csv"
    ),
    InputData(
        channel_name="test",
        data_source=f"s3://{BUCKET}/{TEST_PREFIX}",
        content_type="text/csv"
    )
]

output_data_config = OutputDataConfig(
    s3_output_path=f"s3://{BUCKET}/{MODEL_PREFIX}"
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    base_job_name="transformadores-sklearn-v3",
    source_code=source_code,
    compute=compute,
    stopping_condition=stopping_condition,
    input_data_config=input_data_config,
    output_data_config=output_data_config,
    hyperparameters={
        "n-estimators": 100,
        "max-depth": 10
    }
)

print("ModelTrainer preparado correctamente.")

## Paso 6. OPCIONAL — Intentar el Training Job administrado

**Esta es la única sección de este notebook que puede crear cómputo administrado.**

La cuenta utilizada en el laboratorio tiene:

```text
ml.m5.large for training job usage = 0
```

Por tanto, se espera `ResourceLimitExceeded` si se intenta ejecutar.

La celda captura ese error para que el notebook no quede bloqueado.

In [26]:
training_job = None
managed_training_available = False

try:
    training_job = trainer.train(wait=True, logs=True)
    managed_training_available = True

    print("Training Job finalizado correctamente.")
    print(f"Nombre: {training_job.job_name}")

except Exception as exc:
    print("El Training Job administrado no pudo ejecutarse.")
    print(f"Tipo de error: {type(exc).__name__}")
    print(f"Detalle: {exc}")
    print()
    print("El laboratorio continuará utilizando los artefactos del Notebook 02.")

## Paso 7. Cierre del Notebook 03 y continuidad

Hay dos escenarios:

### Escenario A — Training Job disponible

El artefacto generado por SageMaker queda en S3 y puede utilizarse en los siguientes pasos.

### Escenario B — Cuota no disponible

Se reutiliza:

```text
S3 / model / mejor_modelo.joblib
```

Esto permite continuar con inferencia, registro del modelo y conceptos de MLOps sin volver a entrenar.

### Resultado pedagógico

Se aprendió la diferencia entre:

- entrenamiento local;
- entrenamiento administrado;
- datos en S3;
- artefactos en S3;
- infraestructura de cómputo;
- cuotas de servicio.

> **El siguiente notebook no dependerá de que el Training Job haya sido exitoso.**